last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [16]:
import os


In [17]:
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

In [18]:
#! curl ipinfo.io

In [19]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

# 🚨 이 부분을 빼먹으셔서 에러가 났던 것입니다! 보안 비밀에서 키를 꺼내 변수에 담아줍니다.
openai_api_key = userdata.get('OPENAI_API_KEY')

# 이제 변수를 api_key 인자에 매칭해 줍니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    api_key=openai_api_key, # 위에서 담은 바구니를 그대로 전달
)

In [20]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.


### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [21]:
!pip install -U "pydantic>=2.0,<=2.11.0"

  Using cached pydantic-2.11.0-py3-none-any.whl.metadata (63 kB)
  Using cached pydantic_core-2.33.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached pydantic-2.11.0-py3-none-any.whl (442 kB)
Using cached pydantic_core-2.33.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.0 MB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-client 0.44.0 requires pydantic>=2.12.5, but you have pydantic 2.11.0 which is incompatible.
google-adk 1.29.0 requires pydantic<3.

In [22]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

In [23]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [24]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [25]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [26]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [27]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [28]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [30]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [31]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [32]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [33]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [34]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [35]:
!pip install tiktoken

In [6]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [37]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [43]:
import os

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [44]:
import openai
from google.colab import userdata

# 1. 코랩 보안 비밀 패널에서 키를 꺼내옵니다.
openai_api_key = userdata.get('OPENAI_API_KEY')

# 2. 꺼내온 키를 OpenAI 클라이언트에 직접 주입해 줍니다.
client = openai.OpenAI(api_key=openai_api_key)

In [45]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [47]:
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings

# 1. 코랩 보안 비밀 패널에서 키 꺼내기
openai_api_key = userdata.get('OPENAI_API_KEY')

# 2. 임베딩 모델을 선언할 때 키를 직접 전달하기
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=openai_api_key  # 🔑 여기에 키를 넣어줍니다!
)

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [49]:
!curl ipinfo.io

{
  "ip": "34.61.6.150",
  "hostname": "150.6.61.34.bc.googleusercontent.com",
  "city": "Council Bluffs",
  "region": "Iowa",
  "country": "US",
  "loc": "41.2619,-95.8608",
  "org": "AS396982 Google LLC",
  "postal": "51502",
  "timezone": "America/Chicago",
  "readme": "https://ipinfo.io/missingauth"
}

In [11]:
# 1. 라이브러리 설치 (이미 하셨다면 생략해도 되지만 안전하게 한 번 더 실행)
!pip install -q sentence_transformers

# 2. 🚨 주소를 langchain_community로 변경해 줍니다!
from langchain_community.embeddings import HuggingFaceEmbeddings

# 3. 모델 로드
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_18308/143846454.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnin

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [52]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [53]:
print(embeddings[1])

[-0.043598730117082596, 0.046768199652433395, -0.026038166135549545, -0.0013704403536394238, 0.06055562198162079, 0.05017256364226341, 0.11730732768774033, -0.027721069753170013, 0.009279526770114899, 0.07751473039388657, 0.001512883580289781, -0.09486886113882065, 0.0018310161540284753, 0.010358039289712906, 0.02260895073413849, 0.043929990381002426, -0.013756091706454754, -0.020579246804118156, -0.0327138714492321, -0.02561490423977375, 0.043547168374061584, 0.08289312571287155, -0.05014246329665184, -0.04123270511627197, -0.05022479221224785, 0.04015067592263222, 0.05747781693935394, 0.012200281023979187, 0.028081471100449562, -0.0607445202767849, -0.0662989392876625, 0.02342132292687893, 0.04859095811843872, -0.015047372318804264, -0.020352834835648537, -0.003334973705932498, 0.03370742127299309, -0.10650115460157394, 0.034283142536878586, 0.06269633769989014, 0.03697563335299492, 0.039933156222105026, 0.060280971229076385, -0.052957531064748764, -0.02507835440337658, 0.06646721810

In [54]:
len(embeddings[1])

384

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [55]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [56]:
query = ["this is red fruit"]

In [57]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7799129958370878
0.602116871556071
0.5388708787265135


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain-chroma

In [2]:
from langchain_chroma import Chroma

In [3]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [7]:
# 🚨 런타임이 재시작되었으므로 이 부품을 다시 불러와야 합니다!
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # (이 아래 줄에 쓰일 녀석도 미리 대비)

# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

# 혹시 tiktoken_len 관련해서도 NameError가 난다면 이 셀 위에 해당 함수 정의 셀을 먼저 실행하셔야 합니다!
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)

In [8]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [9]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma


Chroma에 임베딩 시킵니다  

In [12]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [13]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [14]:
print(docs[0].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [ ]:
!pip install -U langchain langchain-classic

In [16]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [18]:
# 🚨 런타임 재시작으로 날아간 ChatOpenAI 부품을 다시 창고에서 꺼내옵니다!
from langchain_openai import ChatOpenAI
from google.colab import userdata # (혹시 아래에서 api_key=userdata.get...을 쓰신다면 이것도 필수!)

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
openai_api_key = userdata.get('OPENAI_API_KEY') # 🔑 금고에서 키 꺼내기

llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    api_key=openai_api_key       # 🔑 모델에 키 주입하기
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [19]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [20]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_18308/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


마크다운 형식으로 출력해봅니다

In [21]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is somewhat wild-looking, with an interrogative and erratic expression. His features are wayward and obstinate, yet his mouth has something soft and childlike about it. The masculinity and strength are in his eyes and brow, while the lower half of his face is tender, immature, and somewhat feminine. His chin is described as irresolute and boylike, which contradicts the strength of his forehead and expression. His dark brown eyes are full of pride and humility.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [23]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown

# 1. 코랩 보안 비밀 패널에서 키 꺼내기
openai_api_key = userdata.get('OPENAI_API_KEY')

# 2. llm2 객체를 생성할 때 api_key를 직접 매칭해 줍니다!
llm2 = ChatOpenAI(
    model="gpt-4o",
    api_key=openai_api_key  # 🔑 여기에 키를 꼭 넣어주세요!
)

# 3. 질문 던지고 마크다운으로 깔끔하게 출력하기
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))

In Hermann Hesse's novel "Demian," the character Demian is often described in a way that highlights his mysterious and charismatic presence rather than specific physical features. Physically, he is depicted as having a striking appearance that includes:

1. **Eyes**: Demian’s eyes are particularly notable and often emphasized in the novel. They are described as having a penetrating and knowing quality that seems to see into the depths of people's souls.

2. **Face**: His facial features are described as mature and distinct, with an expression that suggests wisdom and experience beyond his years.

3. **Aura**: Beyond physical characteristics, Demian is often depicted as having an aura of confidence and authority, which draws people to him and commands respect.

4. **Age**: Although he is a contemporary of the protagonist, Emil Sinclair, Demian's demeanor is more mature, suggesting a deeper understanding of the world and human nature.

5. **Mannerisms**: Demian's mannerisms are calm and assured, contributing to his enigmatic nature. He has a way of speaking and acting that sets him apart from others and leaves a lasting impression on those who meet him.

Overall, the emphasis in the portrayal of Demian is more on his psychological and spiritual presence than on precise physical descriptions, which supports the novel's themes of self-discovery and the exploration of dualities within the self.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [24]:
# 1. 꼬이지 않는 호환 버전으로 최적화하여 라이브러리 설치
!pip install -U -q langchain langchain-openai langchain-community
!pip install -U -q "pydantic>=2.0,<=2.11.0"
!pip install -U -q tiktoken chromadb

# 2. RAG 구현에 필요한 핵심 모듈 한 번에 불러오기 (Import)
import openai
from google.colab import userdata
from langchain_openai import ChatOpenAI

print("✅ 모든 라이브러리 설치 및 기본 임포트가 완료되었습니다!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00
✅ 모든 라이브러리 설치 및 기본 임포트가 완료되었습니다!


Text splitter 사용을 위한 준비입니다

In [25]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. OpenAI 토큰 수 기준(cl100k_base)으로 길이를 측정하는 함수 정의
def tiktoken_len(text):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    return len(tokens)

# 2. 500토큰 크기로 문서를 쪼개주는 분할기 정의
# (글자 수가 아니라 실제 LLM이 받아들이는 '토큰' 기준으로 정확히 자릅니다.)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    length_function=tiktoken_len
)

print("✅ 토큰 계산 함수 및 Text Splitter 준비 완료!")

✅ 토큰 계산 함수 및 Text Splitter 준비 완료!


### Step 1 Document loader

In [26]:
from langchain_community.document_loaders import PyPDFLoader

# 1. 코랩에 업로드된 데미안 PDF 파일 경로 지정 및 로더 생성
loader = PyPDFLoader("/content/Demian.pdf")

# 2. PDF 파일을 읽어온 뒤, 앞서 정의한 text_splitter를 사용해 토큰 기준으로 쪼갭니다.
# (이 과정을 거치면 원문이 벡터 DB에 저장하기 좋은 크기의 조각들로 분할됩니다.)
docs = loader.load_and_split(text_splitter=text_splitter)

# 3. 잘 쪼개졌는지 확인하기 위해 총 조각 개수와 첫 번째 조각 내용 살짝 보기
print(f"✅ 데미안 PDF가 총 {len(docs)}개의 문서 조각으로 분할되었습니다.")
print("-" * 50)
print("[첫 번째 조각 내용 미리보기]")
print(docs[0].page_content[:200] + "...") # 첫 200자만 출력

✅ 데미안 PDF가 총 182개의 문서 조각으로 분할되었습니다.
--------------------------------------------------
[첫 번째 조각 내용 미리보기]
DEMIAN 
• 
Downloaded from https://www.holybooks.com...


### Step 2 Text splitters

In [27]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 오픈소스 임베딩 모델 로드 (API 키가 필요 없고 코랩 자원으로 직접 계산합니다)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. 쪼개진 문서 조각(docs)들을 임베딩 모델을 거쳐 Chroma Vector DB에 저장합니다.
# (이 과정을 통해 텍스트가 AI가 검색할 수 있는 '수학적 좌표'로 변환됩니다.)
db = Chroma.from_documents(docs, embedding_model)

print("✅ 문서 조각들이 성공적으로 Vector DB(Chroma)에 저장되었습니다!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 문서 조각들이 성공적으로 Vector DB(Chroma)에 저장되었습니다!


### Step 3 Vector Empeddings

In [28]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 오픈소스 임베딩 모델 로드
# (텍스트 조각들을 AI가 이해할 수 있는 수학적 숫자 배열인 '벡터'로 변환해 주는 부품입니다.)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. 쪼개진 문서 조각(docs)들을 임베딩하여 Chroma Vector DB에 저장
# (이 과정을 거치면 '데미안' 책 전체가 검색 가능한 숫자의 형태로 데이터베이스에 차곡차곡 쌓입니다.)
db = Chroma.from_documents(docs, embedding_model)

print("✅ 벡터 임베딩 및 Vector DB(Chroma) 저장 완료!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 벡터 임베딩 및 Vector DB(Chroma) 저장 완료!


### Step 4 Retrievers

In [29]:
# Chroma Vector DB를 검색기(Retriever)로 변환합니다.
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # 가장 관련 있는 조각 3개 추출
)

In [33]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# 1. OpenAI LLM 모델 준비 (gpt-4o-mini가 빠르고 가성비가 좋습니다)
openai_api_key = userdata.get('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key, temperature=0.0)

# 2. 질문에 맞는 본문 조각 찾아오기 (앞서 만든 db 활용)
query = "how Demian look like?"
retrieved_docs = db.similarity_search(query, k=3)

# 3. 찾아온 본문 조각들을 하나로 예쁘게 합치기
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# 4. AI에게 던질 프롬프트(지시문) 구성하기
prompt = ChatPromptTemplate.from_template("""
당신은 소설 <데미안> 전문가입니다. 아래에 제공된 [바탕 지식]만을 참고하여 사용자의 질문에 친절하게 답변해주세요.
반드시 제공된 내용에 기반해야 하며, 소설에 없는 내용을 임의로 지어내지 마세요.

[바탕 지식]
{context}

사용자 질문: {question}
""")

# 5. 최신식 랭체인 파이프라인(LCEL) 구성 및 실행
# (프롬프트에 값을 채우고 -> LLM에 던지고 -> 글자만 쏙 뽑아내는 흐름입니다)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({"context": context, "question": query})

# 6. 결과 화면에 마크다운으로 멋지게 출력하기
print(f"🔍 질문: {query}\n")
print("-" * 50)
print("[🤖 RAG 시스템의 최종 답변]")
display(Markdown(response))

🔍 질문: how Demian look like?

--------------------------------------------------
[🤖 RAG 시스템의 최종 답변]


소설 <데미안>에서 Demian의 외모에 대한 구체적인 묘사는 제공되지 않지만, 주인공이 Demian의 얼굴을 기억하며 그와의 유사성을 느끼는 장면이 있습니다. 주인공은 Demian의 얼굴을 종이에 그린 후, 그 얼굴이 자신과 연결된다고 느끼며, 그 얼굴이 자신의 내면, 운명 또는 다이몬을 표현한다고 생각합니다. 따라서 Demian의 외모는 주인공의 내면 세계와 깊은 연관이 있으며, 그가 찾고 있는 친구의 모습으로 상징화됩니다. Demian은 주인공에게 특별한 의미를 지닌 인물로, 그의 외모는 주인공의 감정과 정체성을 반영하는 존재로 그려집니다.

### Step 5 Question Answering

In [34]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# 1. 모델 및 검색기 설정 (한글 질문에 유리하게 한글 쿼리로 세팅)
openai_api_key = userdata.get('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key, temperature=0.0)

query = "데미안의 외모와 첫인상, 분위기는 어떻게 묘사되어 있나요?"
retrieved_docs = db.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# 2. 프롬프트 구성
prompt = ChatPromptTemplate.from_template("""
당신은 소설 <데미안> 전문가입니다. 아래 제공된 [소설 본문 조각]만을 바탕으로 질문에 답하세요.
본문에 없는 내용을 임의로 상상해서 답변하지 마세요.

[소설 본문 조각]
{context}

질문: {question}
""")

# 3. 체인 생성 및 실행
chain = prompt | llm | StrOutputParser()
response = chain.invoke({"context": context, "question": query})

# 4. 시각화 출력
display(Markdown(f"### ❓ 질문\n{query}"))
print("-" * 60)
display(Markdown(f"### 🤖 RAG 답변\n{response}"))
print("-" * 60)
print("📄 [참조한 실제 본문 조각 (Context)]")
for i, doc in enumerate(retrieved_docs):
    print(f"\n[조각 {i+1}]: {doc.page_content[:150]}...")

### ❓ 질문
데미안의 외모와 첫인상, 분위기는 어떻게 묘사되어 있나요?

------------------------------------------------------------


### 🤖 RAG 답변
데미안의 외모는 "가장 친숙한 지적인 얼굴"과 "결단력 있는 입" 그리고 "넓은 이마의 독특한 광채"로 묘사됩니다. 그는 "연한 색의 맥킨토시"를 입고 있으며, "얇은 지팡이"를 팔에 걸치고 있습니다. 그의 걸음걸이는 "탄력 있는 보폭"과 "곧은 자세"로 나타나며, 이러한 모습은 그가 다가올 때 안정감과 친숙함을 줍니다. 

첫인상과 분위기는 주인공에게 "음악처럼" 들리는 그의 목소리와 함께, "기분 좋은 혼합의 기쁨과 불안"을 느끼게 하며, 그가 주인공에게 주는 "오래된 힘"과 "안정감"이 강조됩니다. 주인공은 데미안을 다시 만난 것에 대해 "모든 것이 다시 제자리를 찾았다"는 느낌을 받습니다.

------------------------------------------------------------
📄 [참조한 실제 본문 조각 (Context)]

[조각 1]: DEMIAN 
.. I don't suppose it is any better with you in Japan. 
The people who don't make a bee-line for the fireside 
are rare the whole world over. ...

[조각 2]: DEMIAN 
.. I don't suppose it is any better with you in Japan. 
The people who don't make a bee-line for the fireside 
are rare the whole world over. ...

[조각 3]: DEMIAN 
.. I don't suppose it is any better with you in Japan. 
The people who don't make a bee-line for the fireside 
are rare the whole world over. ...


In [35]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import sys

# streaming=True 설정을 통해 실시간 출력을 활성화합니다.
openai_api_key = userdata.get('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key, temperature=0.0, streaming=True)

query = "싱클레어에게 에밀 데미안은 어떤 존재인가요?"
retrieved_docs = db.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt = ChatPromptTemplate.from_template("""
아래 본문을 읽고 질문에 대해 깊이 있고 철학적으로 답변해 주세요.
[본문]
{context}

질문: {question}
""")

chain = prompt | llm | StrOutputParser()

print(f"❓ 질문: {query}\n")
print("🤖 실시간 답변 생성 중: ", end="")

# 실시간으로 글자가 생성될 때마다 화면에 바로 뿜어줍니다.
for chunk in chain.stream({"context": context, "question": query}):
    print(chunk, end="")
    sys.stdout.flush()
print("\n\n✅ 출력 완료!")

❓ 질문: 싱클레어에게 에밀 데미안은 어떤 존재인가요?

🤖 실시간 답변 생성 중: 에밀 데미안은 싱클레어에게 단순한 친구나 동료 이상의 존재입니다. 그는 싱클레어의 내면 세계를 탐구하고, 그가 속한 사회와 문화의 본질을 비판적으로 바라보게 하는 철학적 스승이자 멘토의 역할을 합니다. 데미안은 싱클레어에게 개인의 정체성과 자아 발견의 중요성을 일깨우며, 그가 속한 공동체의 허위성과 두려움을 드러냅니다.

데미안이 말하는 '공동체 정신'은 겉으로는 협력과 연대의 상징처럼 보이지만, 실제로는 두려움과 불안에 기반한 집단적 본능에 불과하다는 점을 강조합니다. 이는 싱클레어가 자신의 정체성을 찾고, 사회의 규범과 관습에 의문을 제기하는 데 큰 영향을 미칩니다. 데미안은 싱클레어에게 '자기 자신을 알지 못하는 사람들'의 두려움과 그로 인해 형성된 공동체의 한계를 인식하게 하며, 진정한 자유와 사랑은 개인의 독립적인 기여에서 비롯된다는 메시지를 전달합니다.

결국, 데미안은 싱클레어에게 내면의 진실을 탐구하고, 사회의 규범에 도전하며, 개인의 독창성을 통해 새로운 세계를 창조할 수 있는 가능성을 제시하는 존재입니다. 그는 싱클레어가 두려움과 불안을 극복하고, 진정한 자아를 발견하는 여정에서 필수적인 역할을 하는 인물로, 그가 성장하고 변화하는 데 있어 중요한 촉매제 역할을 합니다. 이러한 점에서 데미안은 단순한 인물이 아니라, 싱클레어의 존재론적 탐구와 자아 발견의 상징적인 인물로 자리 잡고 있습니다.

✅ 출력 완료!


In [36]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

openai_api_key = userdata.get('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key, temperature=0.2)

print("📖 데미안 RAG 지식 챗봇이 가동되었습니다. (종료하려면 'exit' 입력)")
print("-" * 60)

while True:
    user_input = input("✨ 데미안에 대해 무엇이든 물어보세요: ")
    if user_input.lower() == 'exit':
        print("챗봇을 종료합니다. 수고하셨습니다!")
        break

    if not user_input.strip():
        continue

    # 1. 입력된 질문으로 Vector DB 검색
    docs_chunks = db.similarity_search(user_input, k=3)
    current_context = "\n\n".join([d.page_content for d in docs_chunks])

    # 2. 프롬프트 및 체인 실행
    prompt = ChatPromptTemplate.from_template("본문: {context}\n\n질문: {question}\n\n답변:")
    chain = prompt | llm | StrOutputParser()
    bot_response = chain.invoke({"context": current_context, "question": user_input})

    print(f"\n🤖 답변: {bot_response}")
    print("-" * 60)

📖 데미안 RAG 지식 챗봇이 가동되었습니다. (종료하려면 'exit' 입력)
------------------------------------------------------------
✨ 데미안에 대해 무엇이든 물어보세요: 데미안은 남성입니까?

🤖 답변: 네, 데미안은 남성입니다. 본문에서 그의 외모와 행동, 그리고 대화 내용으로 미루어 보아 남성 캐릭터로 묘사되고 있습니다.
------------------------------------------------------------
✨ 데미안에 대해 무엇이든 물어보세요: 데미안은 성인 입니까?

🤖 답변: "데미안"은 헤르만 헤세의 소설로, 주인공인 싱클레어와 데미안의 관계를 통해 성숙과 자기 발견의 과정을 탐구합니다. 데미안은 전통적인 가치관에 도전하고, 개인의 내면을 탐구하는 인물로 묘사됩니다. 그는 싱클레어에게 중요한 멘토이자 영적 안내자로서, 그를 새로운 시각으로 세상을 바라보게 합니다.

따라서 데미안은 성인이라고 단정짓기보다는, 인간 존재의 복잡성과 내면의 진리를 탐구하는 인물로 볼 수 있습니다. 그는 도덕적 기준을 초월한 존재로, 개인의 자유와 자기 실현을 강조하는 역할을 합니다. 이러한 점에서 데미안은 성숙한 인물로 여겨질 수 있지만, 전통적인 의미의 성인과는 다소 다른 개념입니다.
------------------------------------------------------------
✨ 데미안에 대해 무엇이든 물어보세요: 데미안의 직업은?

🤖 답변: "데미안"은 헤르만 헤세의 소설로, 주인공 에밀 싱클레어와 그의 멘토인 카를 데미안의 관계를 중심으로 전개됩니다. 데미안의 직업에 대한 구체적인 언급은 소설 내에서 명확히 드러나지 않지만, 그는 주로 철학적이고 심리적인 통찰을 제공하는 인물로 묘사됩니다. 데미안은 싱클레어에게 삶의 의미와 자아 발견에 대한 깊은 이해를 전달하는 역할을 하며, 그의 존재 자체가 주인공에게 큰 영향을 미칩니다. 따라서 데미안의 직업보다는 그의 사상과 철학이